# 34MLS — Assignment 3: Flow Field Through Porous Media

**Student name:** [Your name here]  
**Student ID:** [Your ID here]  
**Kaggle ID:** [Your Kaggle username here]  
**Deadline:** 11-04-2026 23:59

---

## About this notebook

This notebook is the appendix to the report `assignment3.tex`. It contains the full implementation code, structured in the same order as the report sections.

**Rubric map:**
- Step 1 (this section): Data exploration → feeds report **Section A** (dataset characterization)
- Steps 2–4: Preprocessing, augmentation → feeds report **Section A** (loss functions) and **Section B**
- Steps 5–12: Model implementation, training, comparisons → feeds report **Section C**
- Step 13: Kaggle submission → **Section D**

## Reproducibility seeds

We set seeds at the very top so that every random operation in this notebook — data splits, weight initialization, augmentation — produces the same result every time. This is required by **rubric E** (a peer must be able to reproduce our results).

- `torch.manual_seed` controls PyTorch random operations (weight init, dropout)
- `np.random.seed` controls NumPy random operations (shuffling, splits)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os, json, time

# Reproducibility — must be set before any random operation
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
# CUDA reproducibility -- must be set when a GPU is available
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)          # seed all CUDA devices
    torch.backends.cudnn.deterministic = True  # no non-deterministic cuDNN ops
    torch.backends.cudnn.benchmark     = False # disable auto-tuning (non-deterministic)

# Device — use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
DATA_DIR = 'flow-field-through-a-porous-media-25-26'
os.makedirs('experiments/logs',        exist_ok=True)
os.makedirs('experiments/plots',       exist_ok=True)
os.makedirs('experiments/checkpoints', exist_ok=True)
os.makedirs('submissions',             exist_ok=True)
os.makedirs('fig',                     exist_ok=True)

---
## Step 1 — Data Exploration

**What this step does:** Load all three training files and visualise the data before touching any model.

**Why before anything else?** Report section A requires us to *characterise the dataset in statistical terms* (rubric A, sub-point 3). We cannot write about symmetries or dimensional analysis without first having *seen* what the cross-sections and flow fields look like. This step produces the figures and numbers that will go directly into the report.

**The three files we load:**

| File | What it contains | Shape |
|------|-----------------|-------|
| `train_inputs.npy` | Binary cross-sections: 1 = open pixel, 0 = closed pixel | [1500, 32, 64] |
| `train_params.csv` | Physical parameters per sample: ΔP, L, μ, ΔA | [1500, 5 cols] |
| `train_labels.npy` | Ground truth flow rate field q(x,y) [m/s] | [1500, 32, 64] |

In [ ]:
# Load all training data
train_inputs = np.load(f'{DATA_DIR}/train_inputs.npy')      # binary cross-sections
train_labels = np.load(f'{DATA_DIR}/train_labels.npy')      # flow rate fields q(x,y)
train_params = pd.read_csv(f'{DATA_DIR}/train_params.csv')  # physical parameters

print("=== Shapes ===")
print(f"train_inputs : {train_inputs.shape}  dtype={train_inputs.dtype}")
print(f"train_labels : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"train_params : {train_params.shape}")
print()
print("=== Parameter columns ===")
print(train_params.head())
print()
print("=== Input pixel values (should be 0 and 1 only) ===")
print(f"Unique values in inputs: {np.unique(train_inputs)}")
print()
print("=== Flow field range ===")
print(f"q min:  {train_labels.min():.4e} m/s")
print(f"q max:  {train_labels.max():.4e} m/s")
print(f"q mean: {train_labels.mean():.4e} m/s")

### 1.1 — Example cross-sections and flow fields

**What we plot:** 6 randomly chosen samples. For each: the binary cross-section (left) and the corresponding flow field q(x,y) (right).

**What to look for:**
- The flow field is zero wherever the pixel is closed (=0). You cannot have flow through a wall — this is a hard physical constraint our model must respect.
- The flow field is symmetric in a specific way: if the cross-section has a left-right symmetry, so does q(x,y). This is the equivariance we will discuss in report section A.
- The magnitude of q varies across the open region — pixels near the centre of an open channel flow faster than pixels near the edges (boundary layer effect).

In [ ]:
np.random.seed(SEED)  # ensure same samples every run
sample_ids = np.random.choice(len(train_inputs), size=6, replace=False)

fig, axes = plt.subplots(6, 2, figsize=(10, 18))
fig.suptitle('Example cross-sections and flow fields', fontsize=14, y=1.01)

for row, idx in enumerate(sample_ids):
    # Cross-section (binary image)
    ax_cs = axes[row, 0]
    im0 = ax_cs.imshow(train_inputs[idx], cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax_cs.set_title(f'Sample {idx} — Cross-section')
    ax_cs.set_xlabel('y pixel index')
    ax_cs.set_ylabel('x pixel index')
    plt.colorbar(im0, ax=ax_cs, label='open (1) / closed (0)')

    # Flow field
    ax_fl = axes[row, 1]
    im1 = ax_fl.imshow(train_labels[idx], cmap='viridis', aspect='auto')
    ax_fl.set_title(f'Sample {idx} — Flow field q(x,y) [m/s]')
    ax_fl.set_xlabel('y pixel index')
    ax_fl.set_ylabel('x pixel index')
    plt.colorbar(im1, ax=ax_fl, label='q [m/s]')

plt.tight_layout()
plt.savefig('experiments/plots/exploration_cross_sections_and_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_cross_sections_and_flow.png")

### 1.2 — Physical parameter distributions

**What we plot:** Histograms of the four physical parameters across all 1500 training samples.

**Why this matters for the report:**
- Report section A asks us to *characterise the dataset in statistical terms*.
- The parameter distributions tell us how diverse the training set is — are we training on a narrow range of viscosities, or a wide one?
- The distributions also inform **dimensional analysis**: if ΔP, L, μ, ΔA all vary independently, we must make our model robust to all combinations. A model that hard-codes a specific viscosity will fail on the test set.

**Physical reminder of what each parameter means:**
- **ΔP [Pa]** — pressure drop driving the flow. Higher ΔP → faster flow. q scales linearly with ΔP (Darcy's law).
- **L [m]** — channel length. Longer channel → slower flow (more resistance). q scales as 1/L.
- **μ [Pa·s]** — fluid viscosity. More viscous fluid → slower flow. q scales as 1/μ.
- **ΔA [m²]** — pixel area. Affects the physical size of each open region.

In [ ]:
param_cols = ['delta_p', 'L', 'visc', 'delta_A']
param_labels = [
    r'$\Delta P$ [Pa]',
    r'$L$ [m]',
    r'$\mu$ [Pa$\cdot$s]',
    r'$\Delta A$ [m$^2$]'
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribution of physical parameters across 1500 training samples', fontsize=13)

for ax, col, label in zip(axes.flat, param_cols, param_labels):
    values = train_params[col].values
    ax.hist(values, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count')
    ax.set_title(f'{label}  |  min={values.min():.3e}  max={values.max():.3e}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_parameter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_parameter_distributions.png")

print("\n=== Parameter summary statistics ===")
print(train_params[param_cols].describe().to_string())

### 1.3 — Flow field value distribution

**What we plot:**
1. Histogram of ALL q values across all 1500 samples (one value per pixel per sample = 1500 × 32 × 64 = 3,072,000 values).
2. Distribution of the **total volumetric flow** Q per sample, where Q = Σ q(x,y) · ΔA [m³/s].

**Why:**
- The q histogram will show a large spike at zero (all closed pixels have q=0). The non-zero values tell us the actual flow rate magnitudes we need to predict.
- Q per sample tells us how much the total flow varies across the dataset — this matters for choosing a good loss function.
- These plots go directly into the report (rubric A, dataset characterisation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Flow field statistics across training set', fontsize=13)

# --- Plot 1: all q values (log scale to show the zero spike) ---
ax = axes[0]
all_q = train_labels.flatten()
ax.hist(all_q, bins=80, color='steelblue', edgecolor='none', log=True)
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count (log scale)')
ax.set_title('All q values\n(log-scale count)')
ax.grid(True, alpha=0.3)

# --- Plot 2: non-zero q values only ---
ax = axes[1]
nonzero_q = all_q[all_q > 0]
ax.hist(nonzero_q, bins=80, color='darkorange', edgecolor='none')
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count')
ax.set_title(f'Non-zero q values only\n({len(nonzero_q):,} pixels open)')
ax.grid(True, alpha=0.3)

# --- Plot 3: total volumetric flow Q per sample ---
ax = axes[2]
# Q = sum of q(x,y) * delta_A for each sample
# delta_A varies per sample, so we use the per-sample value from params
delta_A_vals = train_params['delta_A'].values          # shape [1500]
q_sums = train_labels.reshape(len(train_labels), -1).sum(axis=1)  # sum over pixels
Q_per_sample = q_sums * delta_A_vals                   # Q [m³/s] per sample
ax.hist(Q_per_sample, bins=60, color='seagreen', edgecolor='none')
ax.set_xlabel(r'$Q = \sum_{xy} q(x,y) \cdot \Delta A$ [m³/s]')
ax.set_ylabel('Count')
ax.set_title('Total volumetric flow Q per sample')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_flow_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_flow_distribution.png")

print(f"\nNon-zero q pixels: {len(nonzero_q):,} / {len(all_q):,} total ({100*len(nonzero_q)/len(all_q):.1f}% open)")
print(f"Q range: [{Q_per_sample.min():.3e}, {Q_per_sample.max():.3e}] m³/s")
print(f"Q mean:   {Q_per_sample.mean():.3e} m³/s")

### 1.4 — Open pixel fraction per sample

**What we plot:** For each of the 1500 samples, what fraction of the 32×64 = 2048 pixels are open (=1)?

**Why this matters:**
- This tells us how porous the media is on average. A sample with 10% open pixels is very dense (little flow); one with 80% is nearly empty (high flow).
- The distribution of open fractions tells us whether the training set is balanced or biased toward one type of medium.
- It also gives us intuition for the *non-physical output* problem (rubric B): a network predicting q > 0 on closed pixels is physically wrong, regardless of how small the value is.

In [ ]:
# Open fraction = number of pixels with value 1 / total pixels per sample
open_fractions = train_inputs.reshape(len(train_inputs), -1).mean(axis=1)  # shape [1500]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Open pixel fraction across training samples', fontsize=13)

# Histogram
ax = axes[0]
ax.hist(open_fractions, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel('Number of samples')
ax.set_title('Distribution of porosity')
ax.axvline(open_fractions.mean(), color='red', linestyle='--', label=f'Mean = {open_fractions.mean():.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter: porosity vs total flow Q
ax = axes[1]
ax.scatter(open_fractions, Q_per_sample, alpha=0.3, s=8, color='steelblue')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel(r'Total flow $Q$ [m³/s]')
ax.set_title('Porosity vs total volumetric flow')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_porosity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_porosity.png")

print(f"\nOpen fraction — min: {open_fractions.min():.2f}  max: {open_fractions.max():.2f}  mean: {open_fractions.mean():.2f}")

### 1.5 — Exploration summary

What we now know from the data, and how it feeds into the report:

| Observation | Report implication |
|-------------|-------------------|
| q = 0 exactly on closed pixels | Hard physical constraint → non-physical output prevention (rubric B) |
| q scales with ΔP, 1/L, 1/μ | Dimensional analysis → degree-1 homogeneous form (rubric A) |
| Cross-sections have left-right and up-down symmetry potential | Symmetry analysis → equivariance of q(x,y) (rubric A) |
| Parameters vary independently across samples | Model must generalise across all (ΔP, L, μ, ΔA) combinations |
| Porosity varies widely | Loss function must handle both near-zero and large flow magnitudes robustly |
| Input tensor is binary (0/1) — doubles as a spatial mask | Masking strategy: multiply network output by input mask in Step 5 to zero closed-pixel predictions (rubric B) |

**Next step:** Preprocessing pipeline — normalisation, train/validation split, PyTorch DataLoaders.

---
## Step 2 — Preprocessing Pipeline

**→ Satisfies rubric A.1 (dimensionless form) and A.3 (dataset characterisation).**

**What this step does:** Define `make_dim_hom()` (adapted from lab4), compute the physical velocity scale v₀ per sample, normalize labels, split into train/val sets, and wrap everything in PyTorch Datasets and DataLoaders.

**Why v₀ normalisation?** The raw flow fields q(x,y) span many orders of magnitude depending on (ΔP, L, μ, ΔA). A network trained on raw SI values would need to learn to output values ranging from ~10⁻⁸ to ~10² m/s simultaneously — numerically unstable. The physics gives us a natural velocity scale:

$$v_0 = \frac{\Delta P \cdot \Delta A}{\mu \cdot L} \quad [\mathrm{m/s}]$$

This scaling follows from a Buckingham π dimensional analysis of the four governing parameters: ΔP [Pa] = [kg·m⁻¹·s⁻²], ΔA [m²], μ [Pa·s] = [kg·m⁻¹·s⁻¹], L [m]. The unique degree-1 combination that yields [m/s] is ΔP·ΔA/(μ·L), since [Pa·m²/(Pa·s·m)] = [m·s⁻¹]. The linear dependence on ΔP and ΔA and the inverse dependence on μ and L is also consistent with Stokes flow physics (Darcy's law): a larger pressure drop drives faster flow, a larger pixel area represents a wider open passage carrying proportionally more flux, greater viscosity resists motion, and a longer channel offers more friction resistance. The dimensionless ratio q/v₀ is O(1) across all samples — this is verified by the statistics printed below: the max value should be of order 1, not 10⁴ or 10⁻⁴. The network learns q/v₀ from the binary cross-section; at inference we multiply back by v₀ to recover q [m/s]. This adapts the same pattern as `make_dim_hom()` in lab4, which formed dimensionless ratios of pendulum parameters.

**Split strategy:** 80/20 random split with the fixed seed → 1200 train + 300 validation samples. The 80/20 ratio is a standard choice for datasets of this size: 1200 samples provides enough diversity for a CNN to learn spatial features, while 300 validation samples gives a statistically reliable estimate of generalisation error. A 90/10 split would yield only 150 validation samples, making the val loss curve noisy and unreliable as an early-stopping signal. Split is done on raw indices before any tensor conversion to prevent data leakage.

**Batch size:** `BATCH_SIZE = 32` balances gradient quality against memory and training speed. Larger batches give more accurate (lower-variance) gradient estimates per step, but consume more GPU memory and can overshoot sharp loss minima; smaller batches introduce beneficial noise that acts as implicit regularisation. With 1200 training samples and batch size 32, each epoch runs 37–38 gradient steps — enough to observe stable loss curves without excessive memory use.

**Dataset and shapes:** Adapts `TensorData` from lab6. Inputs get a channel dimension added (shape: N×1×32×64) so the CNN sees a standard 1-channel image. Labels are kept as (N×32×64) — no channel dimension — because the ground truth is a spatial map indexed by pixel location, not an image with colour channels. The CNN will output (N×1×32×64); the loss function in Step 6 will squeeze the prediction channel dimension to (N×32×64) before comparing with the label.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Guard: fail immediately if CSV column names don't match expectations
_expected_cols = {'delta_p', 'L', 'visc', 'delta_A'}
assert _expected_cols.issubset(train_params.columns), \
    f"Missing columns in train_params.csv. Expected {_expected_cols}, got {set(train_params.columns)}"

# ---------------------------------------------------------------------------
# 2.0 — make_dim_hom(): velocity scale from physical parameters
# adapted from lab4: make_dim_hom(df)
# changes: accepts arrays instead of a DataFrame; returns v0 [m/s] as the
#          label-normalising velocity scale rather than dimensionless inputs;
#          formula derived from Buckingham π (see Step 3 markdown for full derivation)
# ---------------------------------------------------------------------------
def make_dim_hom(delta_p, L, mu, delta_A):
    """
    Compute the physical velocity scale v0 per sample.

    Derived from Buckingham pi analysis of (delta_p, L, mu, delta_A):
        v0 = delta_p * delta_A / (mu * L)   [m/s]

    Encodes Stokes/Darcy linearity: q scales linearly with ΔP/L and ΔA/μ.
    The dimensionless label q/v0 is O(1) for all samples.

    Parameters
    ----------
    delta_p : array-like, shape (N,)   pressure drop [Pa]
    L       : array-like, shape (N,)   channel length [m]
    mu      : array-like, shape (N,)   dynamic viscosity [Pa·s]
    delta_A : array-like, shape (N,)   pixel area [m²]

    Returns
    -------
    v0 : np.ndarray, shape (N,)        velocity scale [m/s]
    """
    delta_p = np.asarray(delta_p, dtype=np.float64)
    L       = np.asarray(L,       dtype=np.float64)
    mu      = np.asarray(mu,      dtype=np.float64)
    delta_A = np.asarray(delta_A, dtype=np.float64)
    assert (delta_p > 0).all() and (L > 0).all() and (mu > 0).all() and (delta_A > 0).all(), \
        "All physical parameters must be strictly positive (no division-by-zero risk)"
    return delta_p * delta_A / (mu * L)   # [m/s]


# ---------------------------------------------------------------------------
# 2.1 — Compute velocity scale v0 per sample using make_dim_hom()
# ---------------------------------------------------------------------------
v0 = make_dim_hom(
    train_params['delta_p'].values,   # ΔP [Pa]
    train_params['L'].values,         # L  [m]
    train_params['visc'].values,      # μ  [Pa·s]
    train_params['delta_A'].values,   # ΔA [m²]
)   # shape (1500,), [m/s]

print("=== v0 (velocity scale) statistics ===")
print(f"v0 min:  {v0.min():.4e} m/s")
print(f"v0 max:  {v0.max():.4e} m/s")
print(f"v0 mean: {v0.mean():.4e} m/s")

# ---------------------------------------------------------------------------
# 2.2 — Normalise labels by v0
# q_norm[i] = q[i] / v0[i]  →  dimensionless, O(1)
# ---------------------------------------------------------------------------
# broadcast v0 over the 32×64 spatial grid
q_norm = train_labels / v0[:, None, None]   # shape (1500, 32, 64)

print(f"\n=== Normalised q/v0 statistics ===")
print(f"q/v0 min:  {q_norm.min():.4f}")
print(f"q/v0 max:  {q_norm.max():.4f}")
print(f"q/v0 mean: {q_norm.mean():.4f}")
# Sanity check: q/v0 max should be O(1). If >> 10, the v0 formula is wrong.

# ---------------------------------------------------------------------------
# 2.3 — 80/20 train/val split (on indices, before tensor conversion)
# ---------------------------------------------------------------------------
N = len(train_inputs)                             # 1500

# Re-seed PyTorch before randperm so the split indices are always identical
# regardless of how many PyTorch random ops ran earlier in this notebook
# (e.g. accidentally re-running a training cell). This is intentional: the
# train/val partition must be stable across all runs.
torch.manual_seed(SEED)
perm = torch.randperm(N).numpy()                  # reproducible shuffle

n_train = int(0.8 * N)                            # 1200
train_idx = perm[:n_train]                        # shape (1200,)
val_idx   = perm[n_train:]                        # shape (300,)

print(f"\n=== Train/val split ===")
print(f"Train samples: {len(train_idx)}")
print(f"Val   samples: {len(val_idx)}")

# ---------------------------------------------------------------------------
# 2.4 — Convert to float32 tensors and add channel dimension
# inputs:  (N, 32, 64) → (N, 1, 32, 64)   (1-channel image for CNN)
# labels:  (N, 32, 64) — no channel dim; CNN output (N,1,32,64) will be
#          squeezed in the loss function (Step 6) before comparing
# ---------------------------------------------------------------------------
inputs_tensor = torch.tensor(train_inputs, dtype=torch.float32).unsqueeze(1)  # (1500,1,32,64)
labels_tensor = torch.tensor(q_norm,       dtype=torch.float32)               # (1500,32,64)

# Slice by split indices
X_train, y_train = inputs_tensor[train_idx], labels_tensor[train_idx]
X_val,   y_val   = inputs_tensor[val_idx],   labels_tensor[val_idx]
v0_train = torch.tensor(v0[train_idx], dtype=torch.float32)   # [m/s] kept for de-normalisation
v0_val   = torch.tensor(v0[val_idx],   dtype=torch.float32)   # [m/s] kept for de-normalisation

print(f"\nX_train shape: {X_train.shape}   y_train shape: {y_train.shape}")
print(f"X_val   shape: {X_val.shape}   y_val   shape: {y_val.shape}")

# ---------------------------------------------------------------------------
# 2.5 — PyTorch Dataset and DataLoader
# adapted from lab6: TensorData class
# changes: removed device param — tensors stay on CPU here and are moved
#          to device in the training loop via batch.to(device)
# ---------------------------------------------------------------------------
class TensorData(Dataset):
    # adapted from lab6: TensorData
    # changes: removed device param — moved to device in training loop
    def __init__(self, input_tensor, label_tensor):
        self.input  = input_tensor
        self.labels = label_tensor

    def __len__(self):
        return self.input.size(0)

    def __getitem__(self, idx):
        return self.input[idx], self.labels[idx]


BATCH_SIZE = 32

train_dataset = TensorData(X_train, y_train)
val_dataset   = TensorData(X_val,   y_val)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)   # reproducible shuffle
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print(f"\n=== DataLoaders ===")
print(f"Train batches: {len(train_loader)}  (batch size {BATCH_SIZE})")
print(f"Val   batches: {len(val_loader)}")

# Quick sanity check: one batch
xb, yb = next(iter(train_loader))
print(f"\nSample batch — x: {xb.shape}, y: {yb.shape}")
print(f"x values in {{0,1}}: {xb.unique().tolist()}")
print(f"y (q/v0) range in batch: [{yb.min():.4f}, {yb.max():.4f}]")

---
## Step 3 — Dimensional Analysis

**→ Satisfies rubric A.1 (dimensionless and degree-1 homogeneous form).**

**What this step does:** Present the formal Buckingham π derivation of the velocity scale v₀, verify that the second dimensionless group π₂ = ΔA/L² is constant across the training set (confirming it does not need to be passed as a network input), and cross-validate `make_dim_hom()` (defined in Step 2) against independent computation.

**Buckingham π theorem applied to this problem:**

We have 4 physical parameters and 3 independent base units:

| Parameter | Symbol | SI units | Dimensions |
|-----------|--------|----------|------------|
| Pressure drop | ΔP | Pa | kg · m⁻¹ · s⁻² |
| Channel length | L | m | m |
| Dynamic viscosity | μ | Pa·s | kg · m⁻¹ · s⁻¹ |
| Pixel area | ΔA | m² | m² |

The output q has dimensions [m/s] = m · s⁻¹. By the Buckingham π theorem: 4 parameters + 1 output − 3 base units = **2 dimensionless groups**. One group is the target itself: π₁ = q / v₀. The second group is a dimensionless geometric ratio: π₂ = ΔA / L², which compares the pixel area to the square of the channel length. If π₂ were not constant across the dataset, it would need to be passed to the network as an additional scalar input. The code below verifies its distribution numerically. If π₂ is effectively constant (coefficient of variation < 1%), it carries no information and can be safely ignored — the network only needs the binary cross-section as input.

The velocity scale that absorbs π₁:

$$v_0 = \frac{\Delta P \cdot \Delta A}{\mu \cdot L} \quad [\text{m/s}]$$

Dimensional verification: [Pa · m² / (Pa·s · m)] = [kg·m⁻¹·s⁻² · m² / (kg·m⁻¹·s⁻¹ · m)] = [m · s⁻¹] ✓

Note that v₀ itself is degree-0 homogeneous (scale-invariant): scaling all four physical parameters simultaneously by any λ > 0 gives v₀(λΔP, λΔA, λμ, λL) = (λΔP)(λΔA)/((λμ)(λL)) = λ²/λ² · v₀ = v₀. The degree-1 homogeneous form of the learning problem is q = v₀ · f(cross-section): multiplying v₀ by λ multiplies q by the same λ, i.e. q(λv₀, cross-section) = λ · q(v₀, cross-section). Here f is the dimensionless function of the binary geometry that the network learns. The degree-1 homogeneous structure is enforced by the normalisation pipeline: the network receives q/v₀ as its target and outputs a dimensionless field, which is then rescaled by v₀ at inference time to recover physical units.

The derivation is grounded in Stokes/Darcy physics: in the low-Reynolds-number regime, flow rate is linear in pressure gradient (Darcy's law), so q ∝ ΔP/L. Combining this with the geometric scale ΔA/μ gives v₀ = ΔP·ΔA/(μ·L). The `make_dim_hom()` function defined in Step 2 encodes this formula and is cross-validated below.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# 3.1 — Verify second Buckingham π group: π₂ = ΔA / L²
# If π₂ is constant (CV < 1%), it carries no information and the network
# does not need it as an additional scalar input.
# If π₂ varies (CV ≥ 1%), it must be added as a per-sample input — the
# pipeline is halted with an error so the design can be updated.
# ---------------------------------------------------------------------------
pi2 = train_params['delta_A'].values / train_params['L'].values**2   # dimensionless

pi2_mean = pi2.mean()
pi2_std  = pi2.std()
pi2_cv   = pi2_std / pi2_mean * 100   # coefficient of variation [%]

print("=== Second Buckingham π group: π₂ = ΔA / L² ===")
print(f"min  : {pi2.min():.6e}")
print(f"max  : {pi2.max():.6e}")
print(f"mean : {pi2_mean:.6e}")
print(f"std  : {pi2_std:.6e}")
print(f"CV   : {pi2_cv:.4f}%")

if pi2_cv < 1.0:
    print(f"\nπ₂ is effectively constant (CV = {pi2_cv:.4f}% < 1%) ✓")
    print("→ π₂ carries no discriminating information across samples.")
    print("→ The network does NOT need ΔA/L² as an additional scalar input.")
    print("→ The binary cross-section + v₀ normalisation fully specifies the problem.")
else:
    raise ValueError(
        f"π₂ = ΔA/L² varies significantly (CV = {pi2_cv:.2f}% ≥ 1%). "
        "Add π₂ as a per-sample scalar input to the network before proceeding. "
        "Update TensorData and DataLoaders in Step 2 to concatenate π₂ with the cross-section."
    )

# ---------------------------------------------------------------------------
# 3.2 — Cross-validate make_dim_hom() against Step 2 inline v0
# make_dim_hom() is defined in Step 2 (cell above) and used to produce
# the DataLoader tensors. We re-apply it here independently to confirm
# the formula is implemented correctly.
# ---------------------------------------------------------------------------
v0_check = make_dim_hom(
    train_params['delta_p'].values,
    train_params['L'].values,
    train_params['visc'].values,
    train_params['delta_A'].values,
)

assert np.allclose(v0_check, v0), \
    "make_dim_hom() cross-check failed — formula mismatch between Step 2 and Step 3"
print("\nmake_dim_hom() cross-check passed ✓  (Step 2 pipeline uses the correct formula)")

q_over_v0 = train_labels / v0_check[:, None, None]   # shape (1500, 32, 64)

print(f"\n=== Dimensionless field q/v0 ===")
print(f"min : {q_over_v0.min():.4f}")
print(f"max : {q_over_v0.max():.4f}   (should be O(1))")
print(f"mean: {q_over_v0.mean():.4f}")
print(f"std : {q_over_v0.std():.4f}")

# ---------------------------------------------------------------------------
# 3.3 — Verification plots: π₂ distribution + q/v₀ histogram
# ---------------------------------------------------------------------------
nonzero_mask = q_over_v0 > 0
q_over_v0_nz = q_over_v0[nonzero_mask]
per_sample_max = q_over_v0.reshape(len(train_labels), -1).max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dimensional analysis verification', fontsize=12)

# π₂ distribution
ax = axes[0]
ax.hist(pi2, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel(r'$\pi_2 = \Delta A \,/\, L^2$ (dimensionless)')
ax.set_ylabel('Count')
ax.set_title(f'Second π group distribution\nCV = {pi2_cv:.4f}%  ({"constant ✓" if pi2_cv < 1 else "VARIES — add as input"})')
ax.grid(True, alpha=0.3)

# Non-zero q/v₀ histogram
ax = axes[1]
ax.hist(q_over_v0_nz, bins=80, color='steelblue', edgecolor='none')
ax.set_xlabel(r'$q / v_0$ (dimensionless)')
ax.set_ylabel('Count')
ax.set_title(f'Non-zero q/v₀ values\n({len(q_over_v0_nz):,} open pixels)')
ax.grid(True, alpha=0.3)

# Per-sample max q/v₀
ax = axes[2]
ax.hist(per_sample_max, bins=60, color='darkorange', edgecolor='none')
ax.set_xlabel(r'max pixel $q / v_0$ per sample')
ax.set_ylabel('Number of samples')
ax.set_title(f'Per-sample max of q/v₀\nmean={per_sample_max.mean():.3f}  std={per_sample_max.std():.3f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/dim_analysis_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/dim_analysis_verification.png")

---
## Step 4 — Data Augmentation

**→ Satisfies rubric B.1 (data augmentation strategy using identified symmetries, and disambiguation).**

**What this step does:** Implement equivariant augmentation using the symmetry group of the problem, apply it to produce an augmented training set, and verify visually that both input and label transform consistently.

**Symmetry group analysis:**
The full physical symmetry group of Stokes flow through a symmetric channel is the dihedral group D₄ (for square domains) comprising four rotations and four reflections. For this problem the cross-section grid is 32×64 (non-square), so 90-degree rotations map the grid to a 64×32 shape that is incompatible with a fixed CNN input size — `torch.rot90` on a (32×64) tensor produces a (64×32) tensor. The Z₄ rotation subgroup (90°, 180°, 270° rotations) is therefore excluded from the offline augmentation.

The valid symmetry subgroup is **G = V₄ ≅ Z₂ × Z₂** (the Klein four-group), containing four elements:

| Element | Operation on input | Operation on label |
|---------|-------------------|-------------------|
| identity | none | none |
| flip_h | flip last axis (width, 64 px) | flip last axis |
| flip_v | flip second-to-last axis (height, 32 px) | flip second-to-last axis |
| flip_hv | flip both axes (≡ 180° in-plane rotation) | flip both axes |

This covers 4 of the 8 elements of D₄. The 4 excluded elements (90°/270° rotations and their reflections) could be recovered at inference time via group-averaging (GroupAvgModel, Step 10), even though they cannot be used for offline augmentation on a non-square grid.

**Why the label transforms identically:** q(x,y) is a scalar field (z-direction flow rate at each pixel). Flipping the geometry spatially rearranges q values to match the mirrored positions — no sign changes are needed because the velocity component is perpendicular to the 2D cross-section plane. This is equivariance: T(q(x,y)) = q(T(x,y)) for all T ∈ G.

**Disambiguation:** For this problem, no disambiguation is needed. Since the label q(x,y) is a continuous scalar field, flipping the input produces a uniquely determined flipped label — there is no ambiguity about which of several possible labels corresponds to a flipped input. Disambiguation would be needed if the symmetry transformation produced multiple valid labels (e.g. if q were a vector field where flipping reverses a component sign). Here, q is a speed-like scalar (always ≥ 0), so T(q) = q ∘ T⁻¹ has a single unique value at every pixel and no tie-breaking rule is required.

**Effect on dataset size:** Each training sample produces 3 additional augmented samples. The effective training set grows from 1200 to 4800 samples (4×), at zero additional data collection cost.

**Offline vs online augmentation:** This implementation pre-generates all 4 group elements for all samples before training (offline). This ensures that each epoch sees every augmented version of every sample exactly once, giving a systematic and reproducible training signal. The trade-off is that a mini-batch of size 32 may contain multiple orientations of the same original sample, reducing within-batch diversity compared to online augmentation (where a random group element is sampled per sample per epoch). For a dataset of 4800 samples this is acceptable — the DataLoader shuffle ensures orientations of the same sample appear in different epochs anyway.

**Adaptation from lab7 `flip()` and `rot()`:**
- Lab7 `flip()`: `torch.flip(x, [-1])` — horizontal flip only, applied to input only (scalar output task)
- Here: flip applied to **both input and label**; vertical flip and combined flip added; `rot()` excluded due to non-square grid
- The key equivariant adaptation: every transformation is applied identically to the paired (x, y) tensors

In [ ]:
# ---------------------------------------------------------------------------
# 4.1 — Augmentation functions
# adapted from lab7: flip(x) = torch.flip(x, [-1])
# changes: applied to both input and label tensors; vertical flip added;
#          combined flip added; input has channel dim (1,32,64), label is (32,64)
# ---------------------------------------------------------------------------

def flip_h(x, y):
    # adapted from lab7: flip(x) = torch.flip(x, [-1])
    """Horizontal flip (left-right). Flips the last axis (width=64)."""
    return torch.flip(x, [-1]), torch.flip(y, [-1])

def flip_v(x, y):
    # adapted from lab7: flip(x) = torch.flip(x, [-1])
    """Vertical flip (up-down). Flips the second-to-last axis (height=32)."""
    return torch.flip(x, [-2]), torch.flip(y, [-2])

def flip_hv(x, y):
    # adapted from lab7: flip(x) = torch.flip(x, [-1])
    """Both flips combined (180-degree rotation in-plane)."""
    return torch.flip(x, [-2, -1]), torch.flip(y, [-2, -1])


# ---------------------------------------------------------------------------
# 4.2 — Offline augmentation: pre-generate all 4 group elements (G = V₄ ≅ Z₂×Z₂)
# Every symmetry operation is applied to every training sample up front.
# Result: 4 × 1200 = 4800 samples stored in memory (~37 MB — well within limits).
# Named train_dataset_aug / train_loader_aug to distinguish from the
# non-augmented train_dataset / train_loader built in Step 2.
# val_dataset and val_loader are untouched — no augmentation at evaluation time.
# ---------------------------------------------------------------------------

def build_augmented_tensors(X, y):
    """
    Pre-generate all four G = V₄ ≅ Z₂×Z₂ versions of every sample.

    Parameters
    ----------
    X : torch.Tensor, shape (N, 1, 32, 64)
    y : torch.Tensor, shape (N, 32, 64)

    Returns
    -------
    X_aug : torch.Tensor, shape (4N, 1, 32, 64)
    y_aug : torch.Tensor, shape (4N, 32, 64)
    """
    ops = [
        lambda x, y: (x, y),   # identity
        flip_h,
        flip_v,
        flip_hv,
    ]
    X_parts, y_parts = [], []
    for op in ops:
        X_t, y_t = op(X, y)   # torch.flip broadcasts over the batch dim
        X_parts.append(X_t)
        y_parts.append(y_t)
    return torch.cat(X_parts, dim=0), torch.cat(y_parts, dim=0)


X_train_aug, y_train_aug = build_augmented_tensors(X_train, y_train)

print(f"Original train tensors : X {X_train.shape}, y {y_train.shape}")
print(f"Augmented train tensors: X {X_train_aug.shape}, y {y_train_aug.shape}")
print(f"Memory (aug X): {X_train_aug.nbytes / 1e6:.1f} MB")
print(f"Memory (aug y): {y_train_aug.nbytes / 1e6:.1f} MB")

# Use distinct names to make the augmented loader explicit.
# All subsequent training steps (Step 5+) should use train_loader_aug.
train_dataset_aug = TensorData(X_train_aug, y_train_aug)

train_loader_aug = DataLoader(
    train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)
# val_loader is unchanged — no augmentation at evaluation time

print(f"\nTrain dataset (augmented): {len(train_dataset_aug)} samples")
print(f"Val   dataset (original) : {len(val_dataset)} samples (no augmentation)")

xb, yb = next(iter(train_loader_aug))
print(f"Sample batch  — x: {xb.shape}, y: {yb.shape}")


# ---------------------------------------------------------------------------
# 4.3 — Visual verification: one sample × all four group elements
# Correct equivariance: the flow pattern must mirror the geometry exactly
# ---------------------------------------------------------------------------

sample_idx = 0   # fixed index for reproducibility (no random call needed)
x_orig = X_train[sample_idx]   # (1, 32, 64)
y_orig = y_train[sample_idx]   # (32, 64)

ops      = [lambda x, y: (x, y), flip_h, flip_v, flip_hv]
op_names = ['Identity\n(original)', 'flip_h\n(left-right)', 'flip_v\n(up-down)', 'flip_hv\n(both)']

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Step 4 — Equivariant augmentation: all four G = V₄ ≅ Z₂×Z₂ operations', fontsize=13)

for col, (op, name) in enumerate(zip(ops, op_names)):
    x_t, y_t = op(x_orig, y_orig)

    ax_top = axes[0, col]
    ax_top.imshow(x_t.squeeze().numpy(), cmap='binary', origin='lower', aspect='auto')
    ax_top.set_title(name, fontsize=10)
    ax_top.set_xlabel('y-axis (width, 64 px)')
    ax_top.set_ylabel('x-axis (height, 32 px)')

    ax_bot = axes[1, col]
    im = ax_bot.imshow(y_t.numpy(), cmap='viridis', origin='lower', aspect='auto')
    ax_bot.set_xlabel('y-axis (width, 64 px)')
    ax_bot.set_ylabel('x-axis (height, 32 px)')
    plt.colorbar(im, ax=ax_bot, fraction=0.046, pad=0.04, label=r'$q/v_0$')

fig.text(0.01, 0.72, 'Input (cross-section)',   va='center', rotation='vertical', fontsize=11)
fig.text(0.01, 0.28, 'Label (q/v\u2080 field)', va='center', rotation='vertical', fontsize=11)

plt.tight_layout(rect=[0.03, 0, 1, 0.96])   # leave room for the row labels on the left
plt.savefig('experiments/plots/augmentation_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/augmentation_verification.png")
print()
print("Visual check: open pixel patterns and q/v0 colour maps must mirror each other")
print("consistently across all four operations — any mismatch indicates a bug.")

---
## Step 5 — Baseline CNN

**→ Satisfies rubric C.1 (baseline feedforward CNN architecture description and implementation) and rubric C.2 (code snippets of key architectural elements).**

**What this step does:** Implement a baseline VGG-like CNN that maps a binary cross-section (1×32×64) to a predicted flow field (32×64) of the same spatial size, and wrap it with a physics mask that enforces q=0 at all closed pixels.

**Adaptation from lab7 `create_model()`:**
Lab7's `create_model()` produces a scalar output via three max-pooling stages followed by a fully-connected head (Flatten → Linear → Linear → Linear → 1). This scalar design is incompatible with our task, where the output is a dense spatial field of 2048 values. The FC head is therefore removed entirely and replaced with bilinear upsampling back to 32×64 followed by a 1×1 conv that collapses channels to a single output channel. To keep the latent spatial resolution from becoming too small, the third encoder block also drops its MaxPool stage (two pooling stages downsample to 8×16; a third would give 4×8 — too coarse). Everything else from lab7 is preserved: the three-block structure of 3×Conv2d+ReLU, the 3×3 kernels with padding=1 (preserving spatial size within each block), and `bias=False` for conv layers (reduces parameters; with normalised O(1) inputs bias terms offer no expressive advantage over the learned weight scales). The output 1×1 conv uses `bias=True` so the network can learn a non-zero per-pixel output mean, which aids convergence when flow fields have non-zero mean.

**Why `num_channels=16`:** The bottleneck spatial size is 8×16. At C=16 the bottleneck tensor holds 16×8×16 = 2048 values — exactly one per output pixel — giving enough representational capacity while keeping the total parameter count to ~18,600, well below the 120,000 bonus cap. Increasing to C=32 roughly quadruples the parameter count (~74k) for marginal capacity gain at the bottleneck; C=8 risks underfitting complex flow patterns.

**Why this architecture is VGG-like:** VGG networks are characterised by repeated blocks of same-size convolutions followed by spatial pooling. Here each of the first two encoder blocks uses three 3×3 convolutions with ReLU activation before a MaxPool — exactly the VGG pattern. Block 3 acts as a latent-space refinement without further downsampling.

**Non-physical outputs and the masking fix:**
Two types of non-physical output are possible from a naive CNN:
1. **Negative flow at open pixels** — q(x,y) is a speed, always ≥ 0. The backbone has no output activation, so a randomly-initialised model will produce negative values (visible in the smoke test range print below). This is expected pre-training; the loss will push predictions towards the true non-negative values during training. Adding a `nn.Softplus()` output activation to enforce non-negativity is deferred to Step 9 (overfitting prevention), where it is demonstrated to improve training stability.
2. **Non-zero flow at closed pixels** — at pixels where the cross-section value is 0, the wall is solid and q must be exactly 0. This is a hard physical constraint. We enforce it by multiplying the backbone output by the binary input mask: `output × cross_section`. This is implemented in the `MaskedCNN` wrapper class below, adding zero extra parameters.

**v0 and dimensional units — inference path:**
Training labels are stored as dimensionless q/v0 (normalised in Step 2, where v0 = ΔP·ΔA/(μ·L)). The CNN therefore predicts q/v0. At inference (Step 13, Kaggle submission), predictions must be multiplied by the per-sample v0 computed from the hidden test parameters to recover physical units [m/s]:
```
q_physical = model(x_test) * v0_test[:, None, None]
```
The tensors `v0_train` and `v0_val` (stored in Step 2) are used in the same way during validation.

**Architecture:**

```
Input:  (B, 1, 32, 64)
Block 1: Conv2d(1→C)+ReLU, Conv2d(C→C)+ReLU, Conv2d(C→C)+ReLU → MaxPool(2,2)  → (B, C, 16, 32)
Block 2: Conv2d(C→C)+ReLU, Conv2d(C→C)+ReLU, Conv2d(C→C)+ReLU → MaxPool(2,2)  → (B, C,  8, 16)
Block 3: Conv2d(C→C)+ReLU, Conv2d(C→C)+ReLU, Conv2d(C→C)+ReLU  [no pool]      → (B, C,  8, 16)
Upsample ×2 (bilinear, align_corners=False)                                      → (B, C, 16, 32)
Upsample ×2 (bilinear, align_corners=False)                                      → (B, C, 32, 64)
Conv2d(C→1, kernel=1×1, bias=True)                                               → (B, 1, 32, 64)
[MaskedCNN.forward] out.squeeze(1)  [removes singleton channel dim safely]       → (B, 32, 64)
× binary mask x.squeeze(1)          [MaskedCNN wrapper — zero extra params]      → (B, 32, 64)
```

Default `num_channels=16`, conv layers `bias=False`, output conv `bias=True`. At C=16 the parameter count is approximately 18,617 — well within the 120,000 bonus cap.

In [ ]:
import torch.nn as nn

# ---------------------------------------------------------------------------
# num_params() — adapted from Assignement_3_helper_functions.ipynb
# Adaptation: added `return total` so callers can save the count programmatically.
# Required for rubric D bonus proof (≤120,000 params evidenced per-layer).
# ---------------------------------------------------------------------------
def num_params(model):
    layers_store = []
    layer_idx = 0
    for param in model.parameters():
        if len(param.shape) > 1:   # weights
            layers_store.append({"layer": layer_idx, "weight": param.numel(), "bias": 0})
            layer_idx += 1
        else:                      # bias (belongs to previous layer)
            layers_store[-1]["bias"] = param.numel()
    for layer in layers_store:
        print(f"  Layer {layer['layer']}: {layer['weight']:,} weights, {layer['bias']} biases")
    total = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total:,}")
    return total


# ---------------------------------------------------------------------------
# 5.1 — Baseline CNN backbone
# Adapted from lab7 create_model() — same three-block VGG-like encoder
# (3×Conv2d+ReLU per block, 3×3 kernels, padding=1, bias=False).
# Key differences from lab7:
#   - Block 3 has NO MaxPool (two pooling stages is sufficient; three would
#     reduce spatial size to 4×8 which is too coarse for a 32×64 output).
#   - FC head completely removed; replaced with bilinear upsample ×2 ×2 + 1×1 conv.
#   - Output conv uses bias=True to allow the network to learn a non-zero mean.
#   - No nn.Flatten in the Sequential; channel squeeze is handled by MaskedCNN.forward().
# ---------------------------------------------------------------------------

def create_model(num_channels=16, bias=False):
    """
    Baseline VGG-like CNN backbone: (B,1,32,64) -> (B,1,32,64).

    Three encoder blocks (3xConv2d+ReLU; first two followed by MaxPool) downsample
    32x64 -> 8x16. Two bilinear upsample stages restore 8x16 -> 32x64. A final 1x1
    conv collapses to 1 output channel. Channel squeeze and physics masking are
    handled by MaskedCNN.forward(), not inside this Sequential.

    Parameters
    ----------
    num_channels : int   number of convolutional feature maps (default 16)
    bias : bool          whether conv layers use a bias term (default False);
                         the output 1x1 conv always uses bias=True.
    """
    C = num_channels

    def conv_relu(in_ch, out_ch):
        return [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=bias),
            nn.ReLU(),
        ]

    return nn.Sequential(
        # --- Encoder block 1: (B,1,32,64) -> (B,C,16,32) ---
        *conv_relu(1, C),
        *conv_relu(C, C),
        *conv_relu(C, C),
        nn.MaxPool2d(2, 2),

        # --- Encoder block 2: (B,C,16,32) -> (B,C,8,16) ---
        *conv_relu(C, C),
        *conv_relu(C, C),
        *conv_relu(C, C),
        nn.MaxPool2d(2, 2),

        # --- Encoder block 3: (B,C,8,16) -> (B,C,8,16)  [no MaxPool] ---
        *conv_relu(C, C),
        *conv_relu(C, C),
        *conv_relu(C, C),

        # --- Decoder: upsample back to 32x64 ---
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),  # -> (B,C,16,32)
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),  # -> (B,C,32,64)

        # --- Output: 1x1 conv collapses channels -> (B,1,32,64) ---
        # bias=True on output layer so the network can learn a non-zero mean output
        nn.Conv2d(C, 1, kernel_size=1, bias=True),
    )


# ---------------------------------------------------------------------------
# 5.2 — Physics mask wrapper
# Multiplies the backbone output by the binary cross-section (the input mask).
# At closed pixels (value = 0), this forces the predicted flow rate to exactly 0,
# enforcing the hard physical constraint: no fluid passes through a solid wall.
# Zero extra parameters added -- the mask is derived from the input at runtime.
#
# NOTE: out.squeeze(1) removes the singleton channel dim from the backbone
# output (B,1,32,64) → (B,32,64). Using squeeze(1) is safer than
# nn.Flatten(start_dim=1, end_dim=2) because it explicitly targets only the
# channel dimension and raises an error if it is not exactly size 1.
# ---------------------------------------------------------------------------

class MaskedCNN(nn.Module):
    """
    Physics-constrained wrapper: enforces q=0 at all closed pixels.

    Forward pass: backbone(x).squeeze(1) * x.squeeze(1)
    The binary cross-section x in {0,1}^(Bx1x32x64) serves as the mask.
    Closed pixels (x=0) produce exactly zero output regardless of backbone weights.

    WARNING: kaggle_loss() uses (truth + eps) in the denominator. At closed
    pixels truth=0, so the denominator is eps=1e-6. Calling kaggle_loss()
    with unmasked predictions would produce loss values ~|pred|/1e-6, which
    are ~1e6 times larger than typical open-pixel terms and will corrupt
    training. Always call kaggle_loss() on MaskedCNN output (or any other
    correctly masked tensor). See Step 6 for details.

    Parameters
    ----------
    backbone : nn.Module  spatial backbone with signature (B,1,32,64) -> (B,1,32,64)
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        out  = self.backbone(x)    # (B, 1, 32, 64)
        out  = out.squeeze(1)      # (B, 32, 64) — removes singleton channel dim safely
        mask = x.squeeze(1)        # (B, 32, 64) — binary {0, 1}
        return out * mask          # zero out all closed pixels


# ---------------------------------------------------------------------------
# 5.3 — Smoke test: verify shapes, masking behaviour, and parameter count
# ---------------------------------------------------------------------------

model = MaskedCNN(create_model(num_channels=16, bias=False))
model.to(device)
model.eval()   # set to evaluation mode -- important once BN/Dropout layers are added

# Shape check: all-zero (all-closed) input
dummy_input = torch.zeros(4, 1, 32, 64).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)

print(f"Input  shape : {tuple(dummy_input.shape)}")
print(f"Output shape : {tuple(dummy_output.shape)}   (expected: (4, 32, 64))")
assert dummy_output.shape == (4, 32, 64), \
    f"Shape mismatch: expected (4, 32, 64), got {tuple(dummy_output.shape)}"

# Masking check: all-closed input -> all-zero output (closed-pixel constraint)
assert dummy_output.abs().max().item() == 0.0, \
    "Masking failed: all-zero input should produce all-zero output"
print("Masking check: all-closed input -> all-zero output (q=0 at closed pixels) [OK]")

# All-open input: output shows backbone range before training
# Negative values are expected at this stage (random initialisation).
# They will be pushed to non-negative values during training via the loss.
# A Softplus output activation to hard-enforce non-negativity is added in Step 9.
open_input = torch.ones(1, 1, 32, 64).to(device)
with torch.no_grad():
    open_output = model(open_input)
print(f"All-open input: output range [{open_output.min():.4f}, {open_output.max():.4f}]  "
      f"(negatives expected pre-training)")

# Parameter count (rubric D bonus proof via num_params helper)
print(f"\n=== Parameter count (num_params) ===")
n_total = num_params(model)
print(f"\nBonus cap (<=120,000): {'PASS [OK]' if n_total <= 120_000 else 'FAIL -- reduce num_channels'}")

# Persist parameter count to experiments/logs/ so it is part of the run record
params_log = {
    "model": "MaskedCNN(create_model(num_channels=16, bias=False))",
    "total_params": n_total,
    "bonus_cap": 120000,
    "bonus_eligible": n_total <= 120000,
}
params_log_path = "experiments/logs/cnn_baseline_params.json"
with open(params_log_path, 'w') as f:
    json.dump(params_log, f, indent=2)
print(f"Parameter count saved to {params_log_path}")

---
## Step 6 — Loss function

**→ Satisfies rubric A.4 (loss function proposal and physical criticism) and rubric C (training with the correct evaluation metric).**

**What this step does:** Implement the Kaggle evaluation metric as a PyTorch loss function so that the same formula is used for both training and evaluation.

**The Kaggle formula (assignment3.tex, eq. 3):**

$$E = \frac{1}{|\mathrm{dataset}|} \sum_{i} \frac{1}{N_X N_Y} \sum_{xy} \left| \frac{q_i^T(x,y) - q_i(x,y)}{q_i^T(x,y) + \varepsilon} \right|$$

This is a **Mean Absolute Relative Error (MARE)**: for each pixel the absolute difference between prediction and ground truth is divided by the ground truth plus a small $\varepsilon$ that prevents division by zero when $q^T \approx 0$. The result is averaged over all pixels and all samples in the batch.

**Adaptation from lab7 `rmsre()`:**
Lab7 defines `rmsre(pred, truth) = ((pred - truth) / truth).square().mean().sqrt()`, which is a Root Mean Squared Relative Error with no $\varepsilon$ guard. Two changes are needed to match the Kaggle formula: (1) replace `/ truth` with `/ (truth + eps)` to add the $\varepsilon$ guard; (2) replace `.square().mean().sqrt()` with `.abs().mean()` because Kaggle uses absolute value (MAE-style), not squared (RMS-style).

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 — Kaggle loss function
# adapted from lab7: rmsre(pred, truth) = ((pred - truth) / truth).square().mean().sqrt()
# changes: (1) denominator truth -> truth + eps  (prevents div-by-zero at q=0 pixels)
#          (2) .square().mean().sqrt() -> .abs().mean()  (Kaggle uses MAE-style, not RMS)
# ---------------------------------------------------------------------------

# eps = 1e-6: chosen to be several orders of magnitude smaller than the
# smallest non-zero q/v0 value in the training set (~0.001, from Step 3
# statistics). This means the eps guard only activates near truly-zero pixels
# and does not meaningfully alter the loss at open pixels where q/v0 >> 1e-6.
# If labels are ever used in SI units (m/s, range 1e-4..1e-2), eps must be
# re-evaluated — at 1e-6 it would still be safe (SI min ~1e-4 >> 1e-6).
EPSILON = 1e-6

def kaggle_loss(pred, truth, eps=EPSILON):
    """
    Mean Absolute Relative Error -- matches Kaggle's E formula (assignment3.tex eq. 3).

    E = mean( |pred - truth| / (truth + eps) )

    Parameters
    ----------
    pred  : torch.Tensor  predicted flow field (B, 32, 64) or (B, 1, 32, 64)
    truth : torch.Tensor  ground truth flow field, same shape as pred
    eps   : float         epsilon guard against division by zero (default 1e-6)

    Returns
    -------
    torch.Tensor  scalar loss value E

    WARNING — closed pixels: at pixels where the cross-section is 0, ground
    truth q=0 and the denominator becomes eps=1e-6. If `pred` is unmasked
    (backbone output without MaskedCNN), closed-pixel predictions can be
    non-zero and each such term contributes |pred|/1e-6 ~ 1e6 × |pred| to
    the loss, silently exploding training. Always pass MaskedCNN output to
    this function so that closed-pixel predictions are exactly 0.
    """
    return ((pred - truth).abs() / (truth + eps)).mean()


# ---------------------------------------------------------------------------
# 6.2 — Sanity checks
# ---------------------------------------------------------------------------

torch.manual_seed(SEED)
dummy_truth = torch.rand(4, 32, 64).abs()   # non-negative ground truth
dummy_pred  = dummy_truth.clone()

# Check 1: perfect predictions -> E = 0
e_perfect = kaggle_loss(dummy_pred, dummy_truth).item()
assert abs(e_perfect) < 1e-9, f"Perfect prediction should give E=0, got {e_perfect}"
print(f"Perfect prediction  -> E = {e_perfect:.2e}  [expected 0]  [OK]")

# Check 2: all-zero prediction against non-zero truth -> E > 0
e_zero = kaggle_loss(torch.zeros_like(dummy_truth), dummy_truth).item()
assert e_zero > 0, "All-zero prediction should give E > 0"
print(f"All-zero prediction -> E = {e_zero:.4f}  [expected > 0]  [OK]")

# Check 3: constant-factor over-prediction -> E is proportional to factor - 1
factor = 2.0
e_factor = kaggle_loss(factor * dummy_truth, dummy_truth).item()
expected = (factor - 1.0)   # |(2q - q)| / (q + eps) ≈ 1.0 when eps << q
print(f"2x over-prediction  -> E = {e_factor:.4f}  [expected ~{expected:.4f}]  [OK]")

print("\nkaggle_loss() is ready -- use as loss_fn in the training loop (Step 7).")